In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
FWD_MS_COL = "forward_ms"
BWD_MS_COL = "backward_ms"
FWD_MEM_COL = "forward_memory_mb"
BWD_MEM_COL = "backward_memory_mb"


CONV_COL = "conv_type"
DATASET_COL = "dataset"
BACKEND_COL = "backend"
HEADS_COL = "heads"
FEATURE_DIM_COL = "feature_dim"

In [3]:
graph_transformer_out = "/home/fvelikon/projects/cuda_exp/out/gt_new"
gatv2_out = "/home/fvelikon/projects/cuda_exp/out/test_graph_reordering"
gcn_out = "/home/fvelikon/projects/cuda_exp/out/gcn"

In [4]:
REORDERING_COL = "graph_reordering_partition_size"

# other columns used for tuning parameters

In [ ]:
BASELINE = "dgl"


# def add_speedup_columns(df_in: pd.DataFrame,
#                         idx_cols:list[str]=[DATASET_COL, FEATURE_DIM_COL, HEADS_COL],
#                         baseline: str = BASELINE) -> pd.DataFrame:
#     """
#     For each (dataset, feature_dim, heads) triple, compute speedup
#     vs `baseline` separately for forward_ms and backward_ms.
#     """
#     df = df_in.copy()

#     for metric in [FWD_MS_COL, BWD_MS_COL]:
#         base_col = f"{metric}_baseline_{baseline}"
#         speed_col = f"{metric}_speedup_vs_{baseline}"
#         df[base_col] = np.nan
#         df[speed_col] = np.nan

#         for key, group in df.groupby(idx_cols):
#             base_rows = group[group["backend"] == baseline]
#             if base_rows.empty:
#                 continue
#             base_val = base_rows.iloc[0][metric]
#             if not np.isfinite(base_val) or base_val <= 0:
#                 continue

#             mask = (df["dataset"] == key[0]) & (df["feature_dim"] == key[1]) & (df["heads"] == key[2])
#             df.loc[mask, base_col] = base_val
#             df.loc[mask, speed_col] = base_val / df.loc[mask, metric]

#     return df

# BASELINE = "dgl"

def add_speedup_columns(df_in: pd.DataFrame, baseline: str = BASELINE) -> pd.DataFrame:
    """
    For each (dataset, feature_dim, heads) triple, compute speedup
    vs `baseline` separately for forward_ms and backward_ms.
    """
    df = df_in.copy()
    idx_cols = [CONV_COL, DATASET_COL, FEATURE_DIM_COL, HEADS_COL, REORDERING_COL]

    for metric in [FWD_MS_COL, BWD_MS_COL, FWD_MEM_COL, BWD_MEM_COL]:
        base_col = f"{metric}_baseline_{baseline}"
        speed_col = f"{metric}_speedup_vs_{baseline}"
        df[base_col] = np.nan
        df[speed_col] = np.nan

        for key, group in df.groupby(idx_cols):
            base_rows = group[group["backend"] == baseline]
            if base_rows.empty:
                continue
            base_val = base_rows.iloc[0][metric]
            if not np.isfinite(base_val) or base_val <= 0:
                continue

            mask = (
                (df[CONV_COL] == key[0]) &
                (df[DATASET_COL] == key[1]) &
                (df[FEATURE_DIM_COL] == key[2]) &
                (df[HEADS_COL] == key[3]) &
                (df[REORDERING_COL] == key[4])
            )
            df.loc[mask, base_col] = base_val
            df.loc[mask, speed_col] = base_val / df.loc[mask, metric]

    return df


In [33]:
a = pd.read_csv(gcn_out, index_col=0)
a[HEADS_COL] = 1

df = pd.concat([
    pd.read_csv(gatv2_out, index_col=0),
    pd.read_csv(graph_transformer_out, index_col=0),
    a
])

In [34]:
df = add_speedup_columns(df, BASELINE)

forward_ms
backward_ms
forward_memory_mb
backward_memory_mb


In [35]:
df

,conv_type,dataset,backend,num_nodes,num_edges,avg_node_degree,feature_dim,heads,grad_A_reduce_row_chunk_size,forward_ms,...,registers_per_sm,experiment_name,forward_ms_baseline_dgl,forward_ms_speedup_vs_dgl,backward_ms_baseline_dgl,backward_ms_speedup_vs_dgl,forward_memory_mb_baseline_dgl,forward_memory_mb_speedup_vs_dgl,backward_memory_mb_baseline_dgl,backward_memory_mb_speedup_vs_dgl
0,gat_v2,artnet-exp,cuda,50405,560696,0.089897,256,2,16.0,9.248563,...,65536,gat_v2_cuda_artnet-exp_feature_dim_256_heads_2...,8.892621,0.961514,14.359348,1.411668,3589.371582,8.416658,5083.212402,5.471114
1,gat_v2,artnet-exp,cuda,50405,560696,0.089897,256,4,16.0,7.085261,...,65536,gat_v2_cuda_artnet-exp_feature_dim_256_heads_4...,17.948672,2.533241,27.704218,1.099364,7083.642578,9.676778,9766.990723,6.021433
2,gat_v2,artnet-exp,cuda,50405,560696,0.089897,256,8,16.0,14.158438,...,65536,gat_v2_cuda_artnet-exp_feature_dim_256_heads_8...,35.410739,2.501034,57.128857,1.375664,14060.958008,10.607544,19142.161133,6.367965
3,gat_v2,artnet-exp,cuda,50405,560696,0.089897,256,2,32.0,3.640115,...,65536,gat_v2_cuda_artnet-exp_feature_dim_256_heads_2...,8.892621,2.442950,14.359348,1.181146,3589.371582,8.259300,5083.212402,5.471114
4,gat_v2,artnet-exp,cuda,50405,560696,0.089897,256,4,32.0,7.101645,...,65536,gat_v2_cuda_artnet-exp_feature_dim_256_heads_4...,17.948672,2.527397,27.704218,1.295615,7083.642578,9.697348,9766.990723,6.030092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,gcn,web-traffic,torch_native_gcn,2890331,12895369,0.224137,512,1,NaN,17.601537,...,65536,gcn_torch_native_gcn_web-traffic_feature_dim_512,33.813504,1.921054,57.185382,0.834064,36292.883301,1.199733,47627.149414,1.003589
224,gcn,web-traffic,triton_block_sparse,2890331,12895369,0.224137,64,1,NaN,17.447935,...,65536,gcn_triton_block_sparse_web-traffic_feature_di...,5.671321,0.325043,19.427737,0.814798,16532.883301,1.069475,17987.149414,1.002955
225,gcn,web-traffic,triton_block_sparse,2890331,12895369,0.224137,128,1,NaN,20.846387,...,65536,gcn_triton_block_sparse_web-traffic_feature_di...,9.227879,0.442661,23.939276,1.178402,19356.883301,1.123828,22223.149414,1.002390
226,gcn,web-traffic,triton_block_sparse,2890331,12895369,0.224137,256,1,NaN,27.476376,...,65536,gcn_triton_block_sparse_web-traffic_feature_di...,17.423770,0.634136,34.500711,1.280620,24999.238770,1.204711,30686.682617,1.001730


In [36]:
import math
import numpy as np
import matplotlib.pyplot as plt


def plot_all_speedups_by_dataset(
    df_speed: pd.DataFrame,
    baseline: str = BASELINE,
):
    """
    For each (heads, feature_dim) combination:
      - For each metric in {forward/backward time, forward/backward memory}:
          * Create a figure with subplots (one per dataset).
          * Each subplot: bar chart of speedup vs baseline for all non-baseline backends.

    Assumes `add_speedup_columns` has already been called on df_speed.
    """

    metrics = [FWD_MS_COL, BWD_MS_COL, FWD_MEM_COL, BWD_MEM_COL]
    metric_pretty = {
        FWD_MS_COL: "Forward time",
        BWD_MS_COL: "Backward time",
        FWD_MEM_COL: "Forward memory",
        BWD_MEM_COL: "Backward memory",
    }

    datasets = sorted(df_speed["dataset"].unique())
    heads_list = sorted(df_speed["heads"].unique())
    dims_list = sorted(df_speed["feature_dim"].unique())

    for h in heads_list:
        for D in dims_list:
            df_hd = df_speed[(df_speed["heads"] == h) & (df_speed["feature_dim"] == D)]
            if df_hd.empty:
                continue

            for metric in metrics:
                speed_col = f"{metric}_speedup_vs_{baseline}"
                if speed_col not in df_hd.columns:
                    continue

                # Only consider rows where speedup is finite
                df_metric = df_hd[np.isfinite(df_hd[speed_col])]
                if df_metric.empty:
                    continue

                # Subplot layout: one subplot per dataset
                n_datasets = len(datasets)
                ncols = min(3, n_datasets)
                nrows = math.ceil(n_datasets / ncols)

                fig, axes = plt.subplots(
                    nrows=nrows,
                    ncols=ncols,
                    figsize=(4 * ncols, 3 * nrows),
                    squeeze=False,
                    sharey=True,
                )

                axes_flat = axes.ravel()

                for ax_idx, dataset in enumerate(datasets):
                    ax = axes_flat[ax_idx]
                    df_ds = df_metric[df_metric["dataset"] == dataset]
                    if df_ds.empty:
                        ax.set_visible(False)
                        continue

                    # Backends except baseline
                    backends = [
                        b for b in sorted(df_ds["backend"].unique())
                        if b != baseline
                    ]
                    if not backends:
                        ax.set_visible(False)
                        continue

                    # Align values with backends
                    vals = []
                    for b in backends:
                        row = df_ds[df_ds["backend"] == b]
                        if row.empty:
                            vals.append(np.nan)
                        else:
                            vals.append(float(row.iloc[0][speed_col]))

                    x = np.arange(len(backends))
                    ax.bar(x, vals)
                    ax.set_xticks(x)
                    ax.set_xticklabels(backends, rotation=45, ha="right")

                    ax.axhline(1.0, linestyle="--")
                    ax.set_title(dataset)
                    if ax_idx % ncols == 0:
                        ax.set_ylabel(f"Speedup vs {baseline}")

                # Hide unused axes
                for ax_idx in range(n_datasets, len(axes_flat)):
                    axes_flat[ax_idx].set_visible(False)

                fig.suptitle(
                    f"{metric_pretty.get(metric, metric)} speedup vs {baseline}\n"
                    f"heads={h}, feature_dim={D}",
                    y=0.99,
                )
                fig.tight_layout(rect=[0, 0, 1, 0.95])
                plt.show()


In [ ]:
import pandas as pd
import numpy as np

def build_speedup_table_with_dataset_index(
    df_speed: pd.DataFrame,
    baseline: str,
) -> pd.DataFrame:
    """
    Build a single speedup table with a MultiIndex including dataset.

    Index:
        ('conv_type', 'dataset', 'heads', 'feature_dim')  if 'heads' in df_speed.columns
        ('conv_type', 'dataset', 'feature_dim')          otherwise

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem'} and values are
        speedup vs baseline (>1 = better than baseline).

    Baseline backend column is dropped (speedup is always 1.0).
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
    ]

    # decide index cols
    idx_cols = [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames = []

    for metric, short_name in metrics:
        speed_col = f"{metric}_speedup_vs_{baseline}"
        if speed_col not in df_speed.columns:
            continue

        df_metric = df_speed[np.isfinite(df_speed[speed_col])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=speed_col,
            aggfunc="max",
        )

        # drop baseline backend (always 1.0)
        pivot = pivot.drop(columns=[baseline], errors="ignore")

        if pivot.empty:
            continue

        # attach metric name as top-level column
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    # sort rows & columns
    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table


In [38]:
no_reordering_df = df[df[REORDERING_COL] == -1]
[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]
reordering_df = df[df[REORDERING_COL] != -1]

[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]

[['conv_type',
  'forward_ms',
  'backward_ms',
  'forward_memory_mb',
  'backward_memory_mb',
  'dataset',
  'backend',
  'heads',
  'feature_dim',
  'graph_reordering_partition_size']]

In [41]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"], "dgl").to_markdown(floatfmt=".2f"))

|                                 |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------|----------------------:|-------------------------------------:|-----------------------:|--------------------------------------:|----------------------:|-------------------------------------:|-----------------------:|--------------------------------------:|
| ('gt', 'artnet-exp', 2, 256)    |                  1.04 |                                 1.21 |                   1.13 |                                  0.65 |                  1.21 |                                 1.09 |                   1.10 |                                  1.36 |
| ('gt', 'artnet-exp', 2, 512)    |                  1.02 |                                 1.21 |                   1.08 | 

In [45]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gat_v2"], "dgl").to_markdown(floatfmt=".2f"))

|                                    |   ('Bwd mem', 'cuda') |   ('Bwd time', 'cuda') |   ('Fwd mem', 'cuda') |   ('Fwd time', 'cuda') |
|:-----------------------------------|----------------------:|-----------------------:|----------------------:|-----------------------:|
| ('gat_v2', 'artnet-exp', 2, 256)   |                  5.47 |                   1.60 |                  8.42 |                   2.45 |
| ('gat_v2', 'artnet-exp', 2, 512)   |                  5.47 |                   1.39 |                  8.58 |                   1.96 |
| ('gat_v2', 'artnet-exp', 4, 256)   |                  6.03 |                   1.54 |                  9.70 |                   2.53 |
| ('gat_v2', 'artnet-exp', 4, 512)   |                  6.02 |                   1.50 |                  9.87 |                   1.97 |
| ('gat_v2', 'artnet-exp', 8, 256)   |                  6.37 |                   1.62 |                 10.61 |                   2.59 |
| ('gat_v2', 'artnet-exp', 8, 512)   |   

In [46]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"], "dgl").to_markdown(floatfmt=".2f"))

|                                  |   ('Bwd mem', 'torch_native_gcn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'torch_native_gcn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'torch_native_gcn') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'torch_native_gcn') |   ('Fwd time', 'triton_block_sparse') |
|:---------------------------------|----------------------------------:|-------------------------------------:|-----------------------------------:|--------------------------------------:|----------------------------------:|-------------------------------------:|-----------------------------------:|--------------------------------------:|
| ('gcn', 'artnet-exp', 1, 64)     |                              1.06 |                                 0.98 |                               0.43 |                                  1.42 |                              1.43 |                                 1.17 |                               0.42 |                  

In [15]:
print(build_speedup_table_with_dataset_index(reordering_df, "dgl").to_markdown(floatfmt=".2f"))